# 🏥 Health Risk Predictor

A comprehensive health risk prediction system featuring:
- **Patient Profile Embeddings** using FAISS
- **Risk Categorization** with natural language explanations
- **Guardrails** for safe, validated output

## 1. Setup & Dependencies

In [1]:
# Install required packages (run once)
!pip install numpy pandas scikit-learn sentence-transformers faiss-cpu matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 2. Synthetic Patient Data Generation

In [3]:
def generate_synthetic_patients(n_patients=500, seed=42):
    """Generate synthetic patient health data."""
    np.random.seed(seed)

    data = {
        'patient_id': [f'P{str(i).zfill(4)}' for i in range(n_patients)],
        'age': np.random.randint(18, 85, n_patients),
        'gender': np.random.choice(['Male', 'Female'], n_patients),
        'bmi': np.round(np.random.uniform(16, 45, n_patients), 1),
        'systolic_bp': np.random.randint(90, 180, n_patients),
        'diastolic_bp': np.random.randint(60, 120, n_patients),
        'cholesterol': np.random.randint(120, 320, n_patients),
        'glucose': np.random.randint(70, 200, n_patients),
        'smoking': np.random.choice(['Never', 'Former', 'Current'], n_patients, p=[0.5, 0.3, 0.2]),
        'exercise_freq': np.random.choice(['None', 'Light', 'Moderate', 'Heavy'], n_patients, p=[0.2, 0.3, 0.35, 0.15]),
        'family_history': np.random.choice([0, 1], n_patients, p=[0.6, 0.4]),
        'diabetes': np.random.choice([0, 1], n_patients, p=[0.85, 0.15]),
        'heart_disease': np.random.choice([0, 1], n_patients, p=[0.9, 0.1]),
    }

    df = pd.DataFrame(data)

    # Calculate risk score based on factors
    df['risk_score'] = (
        (df['age'] / 85) * 20 +
        (df['bmi'] > 30).astype(int) * 15 +
        (df['systolic_bp'] > 140).astype(int) * 15 +
        (df['cholesterol'] > 240).astype(int) * 15 +
        (df['glucose'] > 126).astype(int) * 10 +
        (df['smoking'] == 'Current').astype(int) * 15 +
        (df['exercise_freq'] == 'None').astype(int) * 5 +
        df['family_history'] * 10 +
        df['diabetes'] * 10 +
        df['heart_disease'] * 15
    )

    # Add noise and categorize
    df['risk_score'] = np.clip(df['risk_score'] + np.random.normal(0, 5, n_patients), 0, 100)
    df['risk_category'] = pd.cut(df['risk_score'], bins=[0, 25, 50, 75, 100],
                                  labels=['Low', 'Moderate', 'High', 'Critical'])

    return df

print("✅ Function defined!")

✅ Function defined!


In [4]:
# Generate the patient data
patients_df = generate_synthetic_patients(500)
print(f"✅ Generated {len(patients_df)} patient records")
print(f"\n📊 Risk Distribution:")
print(patients_df['risk_category'].value_counts())

✅ Generated 500 patient records

📊 Risk Distribution:
risk_category
Moderate    231
High        195
Low          44
Critical     29
Name: count, dtype: int64


In [5]:
# Preview the data
patients_df.head(10)

,patient_id,age,gender,bmi,systolic_bp,diastolic_bp,cholesterol,glucose,smoking,exercise_freq,family_history,diabetes,heart_disease,risk_score,risk_category
0,P0000,69,Male,42.9,110,69,130,103,Never,Heavy,1,0,0,34.378460,Moderate
1,P0001,32,Male,36.9,136,92,129,110,Never,Heavy,1,1,0,43.360582,Moderate
2,P0002,78,Female,17.4,169,114,134,120,Current,None,0,0,0,48.476040,Moderate
3,P0003,38,Female,38.7,148,109,136,151,Never,Moderate,1,0,0,63.621355,High
4,P0004,41,Male,40.0,173,64,287,127,Never,Light,1,0,0,79.026441,Critical
5,P0005,20,Male,37.8,95,106,265,144,Former,Heavy,0,1,0,52.372745,High
6,P0006,39,Female,39.2,108,108,302,92,Former,Moderate,1,0,1,74.424345,High
7,P0007,70,Male,39.9,146,94,272,197,Never,Moderate,0,0,0,70.041197,High
8,P0008,19,Female,21.4,158,103,178,86,Current,Light,1,0,0,36.410494,Moderate
9,P0009,47,Male,22.8,92,93,197,138,Never,Moderate,1,0,0,27.163234,Moderate


## 3. Patient Profile Embeddings (FAISS)

In [6]:
def create_patient_profile_text(row):
    """Convert patient data to text for embedding."""
    return (f"Patient is {row['age']} years old {row['gender'].lower()}. "
            f"BMI: {row['bmi']}, Blood pressure: {row['systolic_bp']}/{row['diastolic_bp']}. "
            f"Cholesterol: {row['cholesterol']}, Glucose: {row['glucose']}. "
            f"Smoking status: {row['smoking']}, Exercise: {row['exercise_freq']}. "
            f"Family history: {'Yes' if row['family_history'] else 'No'}, "
            f"Diabetes: {'Yes' if row['diabetes'] else 'No'}.")

class PatientEmbeddingStore:
    """Store and search patient embeddings using FAISS."""

    def __init__(self):
        self.embeddings = None
        self.patient_ids = None
        self.index = None
        self.model = None

    def build_index(self, df):
        """Build FAISS index from patient data."""
        try:
            from sentence_transformers import SentenceTransformer
            import faiss

            print("📥 Loading embedding model...")
            self.model = SentenceTransformer('all-MiniLM-L6-v2')

            texts = df.apply(create_patient_profile_text, axis=1).tolist()

            print("🔄 Generating embeddings...")
            self.embeddings = self.model.encode(texts, show_progress_bar=True)
            self.patient_ids = df['patient_id'].tolist()

            dimension = self.embeddings.shape[1]
            self.index = faiss.IndexFlatL2(dimension)
            self.index.add(self.embeddings.astype('float32'))

            print(f"✅ Index built with {len(self.patient_ids)} patients ({dimension}D embeddings)")
            return True
        except ImportError as e:
            print(f"⚠️ {e}. Install: pip install sentence-transformers faiss-cpu")
            return False

    def find_similar(self, patient_data, k=3):
        """Find k most similar patients."""
        if self.index is None:
            return []

        text = create_patient_profile_text(patient_data)
        query_embedding = self.model.encode([text])
        distances, indices = self.index.search(query_embedding.astype('float32'), k)

        return [(self.patient_ids[i], float(d)) for i, d in zip(indices[0], distances[0])]

print("✅ Embedding classes defined!")

✅ Embedding classes defined!


In [7]:
# Build the embedding index
embedding_store = PatientEmbeddingStore()
embedding_store.build_index(patients_df)

📥 Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔄 Generating embeddings...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Index built with 500 patients (384D embeddings)


True

## 4. Input Validation Guardrails

In [8]:
class InputValidator:
    """Validate patient input data."""

    VALID_RANGES = {
        'age': (0, 120),
        'bmi': (10, 70),
        'systolic_bp': (70, 250),
        'diastolic_bp': (40, 150),
        'cholesterol': (80, 400),
        'glucose': (40, 400),
    }

    VALID_CATEGORIES = {
        'gender': ['Male', 'Female'],
        'smoking': ['Never', 'Former', 'Current'],
        'exercise_freq': ['None', 'Light', 'Moderate', 'Heavy'],
    }

    @classmethod
    def validate(cls, data):
        """Validate input data and return errors."""
        errors = []

        for field, (min_val, max_val) in cls.VALID_RANGES.items():
            if field in data:
                val = data[field]
                if not isinstance(val, (int, float)):
                    errors.append(f"{field}: must be numeric")
                elif val < min_val or val > max_val:
                    errors.append(f"{field}: must be between {min_val} and {max_val}")

        for field, valid_vals in cls.VALID_CATEGORIES.items():
            if field in data and data[field] not in valid_vals:
                errors.append(f"{field}: must be one of {valid_vals}")

        for field in ['family_history', 'diabetes', 'heart_disease']:
            if field in data and data[field] not in [0, 1, True, False]:
                errors.append(f"{field}: must be 0/1 or True/False")

        return errors

print("✅ InputValidator defined!")

✅ InputValidator defined!


In [9]:
# Test validation with invalid data
test_invalid = {'age': 150, 'bmi': 25, 'gender': 'Unknown'}
errors = InputValidator.validate(test_invalid)
print("🧪 Testing with invalid data:", test_invalid)
print("❌ Errors found:", errors)

# Test with valid data
test_valid = {'age': 45, 'bmi': 25, 'gender': 'Male'}
errors = InputValidator.validate(test_valid)
print("\n🧪 Testing with valid data:", test_valid)
print("✅ Errors found:", errors if errors else "None!")

🧪 Testing with invalid data: {'age': 150, 'bmi': 25, 'gender': 'Unknown'}
❌ Errors found: ['age: must be between 0 and 120', "gender: must be one of ['Male', 'Female']"]

🧪 Testing with valid data: {'age': 45, 'bmi': 25, 'gender': 'Male'}
✅ Errors found: None!


## 5. Output Guardrails

In [10]:
class OutputGuardrails:
    """Apply guardrails to model output."""

    @classmethod
    def check_confidence(cls, probabilities, threshold=0.6):
        """Check if prediction confidence is sufficient."""
        max_prob = max(probabilities)
        if max_prob < threshold:
            return False, f"⚠️ Low confidence ({max_prob:.1%})"
        return True, f"✅ Confidence: {max_prob:.1%}"

    @classmethod
    def format_output(cls, risk_category, risk_score, confidence, factors):
        """Format output with safety measures."""
        output = f"""
┌──────────────────────────────────────────────────────────────┐
│              HEALTH RISK ASSESSMENT                          │
├──────────────────────────────────────────────────────────────┤
│  Risk Level: {risk_category:10} │ Score: {risk_score:5.1f}%              │
│  {confidence:40}                     │
├──────────────────────────────────────────────────────────────┤
│  Top Contributing Factors:                                   │
"""
        for factor in factors[:5]:
            output += f"│    • {factor:54} │\n"
        output += "└──────────────────────────────────────────────────────────────┘"
        return output

print("✅ OutputGuardrails defined!")

✅ OutputGuardrails defined!


## 6. Risk Prediction Model

In [11]:
class HealthRiskPredictor:
    """Main health risk prediction model."""

    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.label_encoder = LabelEncoder()
        self.feature_columns = None
        self.is_trained = False

    def prepare_features(self, df):
        """Prepare features for training/prediction."""
        df_encoded = df.copy()
        df_encoded['gender_encoded'] = (df_encoded['gender'] == 'Male').astype(int)
        df_encoded['smoking_encoded'] = df_encoded['smoking'].map({'Never': 0, 'Former': 1, 'Current': 2})
        df_encoded['exercise_encoded'] = df_encoded['exercise_freq'].map({'None': 0, 'Light': 1, 'Moderate': 2, 'Heavy': 3})

        self.feature_columns = ['age', 'gender_encoded', 'bmi', 'systolic_bp', 'diastolic_bp',
                                'cholesterol', 'glucose', 'smoking_encoded', 'exercise_encoded',
                                'family_history', 'diabetes', 'heart_disease']

        return df_encoded[self.feature_columns]

    def train(self, df):
        """Train the model on patient data."""
        X = self.prepare_features(df)
        y = self.label_encoder.fit_transform(df['risk_category'])

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        self.model.fit(X_train, y_train)
        self.is_trained = True

        y_pred = self.model.predict(X_test)
        accuracy = (y_pred == y_test).mean()

        return {'accuracy': accuracy, 'X_test': X_test, 'y_test': y_test, 'y_pred': y_pred}

    def predict(self, patient_data):
        """Predict risk for a single patient."""
        if not self.is_trained:
            raise ValueError("Model not trained!")

        errors = InputValidator.validate(patient_data)
        if errors:
            raise ValueError(f"Invalid input: {'; '.join(errors)}")

        df = pd.DataFrame([patient_data])
        X = self.prepare_features(df)

        pred = self.model.predict(X)[0]
        probs = self.model.predict_proba(X)[0]

        risk_category = self.label_encoder.inverse_transform([pred])[0]
        risk_score = probs[pred] * 100

        return {
            'risk_category': risk_category,
            'risk_score': risk_score,
            'probabilities': dict(zip(self.label_encoder.classes_, probs)),
            'feature_importance': dict(zip(self.feature_columns, self.model.feature_importances_))
        }

print("✅ HealthRiskPredictor defined!")

✅ HealthRiskPredictor defined!


In [12]:
# Train the model
predictor = HealthRiskPredictor()
metrics = predictor.train(patients_df)
print(f"✅ Model trained with accuracy: {metrics['accuracy']:.2%}")

✅ Model trained with accuracy: 69.00%


## 7. Natural Language Explanation Generator

In [13]:
class ExplanationGenerator:
    """Generate natural language explanations for predictions."""

    RISK_TEMPLATES = {
        'Low': "Your health profile indicates a LOW risk level. Your vital signs and lifestyle factors are within healthy ranges.",
        'Moderate': "Your health profile shows MODERATE risk. Some factors require attention. Consider lifestyle modifications.",
        'High': "Your assessment indicates HIGH risk. Multiple factors contribute to elevated health risks.",
        'Critical': "Your profile shows CRITICAL risk levels. Multiple significant risk factors detected."
    }

    FACTOR_EXPLANATIONS = {
        'age': lambda v: f"Age ({v} years) - {'significant' if v > 60 else 'moderate' if v > 45 else 'minor'} factor",
        'bmi': lambda v: f"BMI ({v}) - {'obese range' if v > 30 else 'overweight' if v > 25 else 'healthy'}",
        'systolic_bp': lambda v: f"Blood pressure - {'elevated' if v > 140 else 'borderline' if v > 120 else 'normal'}",
        'cholesterol': lambda v: f"Cholesterol ({v}) - {'high' if v > 240 else 'borderline' if v > 200 else 'healthy'}",
        'glucose': lambda v: f"Glucose ({v}) - {'diabetic range' if v > 126 else 'pre-diabetic' if v > 100 else 'normal'}",
    }

    @classmethod
    def generate(cls, prediction, patient_data):
        """Generate explanation for prediction."""
        risk_cat = prediction['risk_category']
        factors = []

        importance = prediction['feature_importance']
        top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]

        for feature, _ in top_features:
            if feature in ['age', 'bmi', 'systolic_bp', 'cholesterol', 'glucose']:
                val = patient_data.get(feature, 0)
                factors.append(cls.FACTOR_EXPLANATIONS[feature](val))

        explanation = cls.RISK_TEMPLATES[risk_cat] + "\n\n📋 Key Factors:\n"
        for f in factors:
            explanation += f"  • {f}\n"

        return explanation

print("✅ ExplanationGenerator defined!")

✅ ExplanationGenerator defined!


## 8. Complete Assessment Pipeline

In [14]:
def run_assessment(patient_data):
    """Run complete health risk assessment."""

    print("=" * 60)
    print("🏥 HEALTH RISK ASSESSMENT")
    print("=" * 60)

    # Step 1: Validate
    print("\n[1/4] 🔍 Validating input...")
    errors = InputValidator.validate(patient_data)
    if errors:
        print(f"❌ Validation failed: {errors}")
        return None
    print("✅ Input valid")

    # Step 2: Predict
    print("\n[2/4] 🤖 Running prediction...")
    prediction = predictor.predict(patient_data)
    print(f"✅ Risk: {prediction['risk_category']}")

    # Step 3: Find similar patients
    print("\n[3/4] 🔎 Finding similar patients...")
    similar = embedding_store.find_similar(patient_data, k=3)
    for pid, dist in similar:
        patient = patients_df[patients_df['patient_id'] == pid].iloc[0]
        print(f"   • {pid}: {patient['risk_category']} risk (distance: {dist:.2f})")

    # Step 4: Generate explanation
    print("\n[4/4] 📝 Generating explanation...")
    explanation = ExplanationGenerator.generate(prediction, patient_data)

    # Format output
    probs = list(prediction['probabilities'].values())
    _, conf_msg = OutputGuardrails.check_confidence(probs)

    factors = [f"{k}: {v:.3f}" for k, v in
               sorted(prediction['feature_importance'].items(), key=lambda x: x[1], reverse=True)[:5]]

    output = OutputGuardrails.format_output(
        prediction['risk_category'], prediction['risk_score'], conf_msg, factors)

    print("\n" + output)
    print("\n" + explanation)

    return prediction

print("✅ Pipeline function defined!")

✅ Pipeline function defined!


## 9. Demo: Run Assessment

In [15]:
# Define a sample patient
sample_patient = {
    'patient_id': 'NEW001',
    'age': 55,
    'gender': 'Male',
    'bmi': 28.5,
    'systolic_bp': 145,
    'diastolic_bp': 92,
    'cholesterol': 245,
    'glucose': 118,
    'smoking': 'Former',
    'exercise_freq': 'Light',
    'family_history': 1,
    'diabetes': 0,
    'heart_disease': 0
}

print("📋 Sample Patient Profile:")
for k, v in sample_patient.items():
    if k != 'patient_id':
        print(f"   {k}: {v}")

📋 Sample Patient Profile:
   age: 55
   gender: Male
   bmi: 28.5
   systolic_bp: 145
   diastolic_bp: 92
   cholesterol: 245
   glucose: 118
   smoking: Former
   exercise_freq: Light
   family_history: 1
   diabetes: 0
   heart_disease: 0


In [16]:
# Run the assessment
result = run_assessment(sample_patient)

🏥 HEALTH RISK ASSESSMENT

[1/4] 🔍 Validating input...
✅ Input valid

[2/4] 🤖 Running prediction...
✅ Risk: High

[3/4] 🔎 Finding similar patients...
   • P0492: Moderate risk (distance: 0.01)
   • P0167: High risk (distance: 0.01)
   • P0210: High risk (distance: 0.02)

[4/4] 📝 Generating explanation...


┌──────────────────────────────────────────────────────────────┐
│              HEALTH RISK ASSESSMENT                          │
├──────────────────────────────────────────────────────────────┤
│  Risk Level: High       │ Score:  45.0%              │
│  ⚠️ Low confidence (45.0%)                                    │
├──────────────────────────────────────────────────────────────┤
│  Top Contributing Factors:                                   │
│    • cholesterol: 0.169                                     │
│    • bmi: 0.156                                             │
│    • systolic_bp: 0.150                                     │
│    • glucose: 0.123                                

## 10. Try Your Own Patient

In [17]:
# Modify these values and run the cell!
my_patient = {
    'patient_id': 'CUSTOM',
    'age': 40,
    'gender': 'Female',
    'bmi': 24.0,
    'systolic_bp': 120,
    'diastolic_bp': 80,
    'cholesterol': 180,
    'glucose': 90,
    'smoking': 'Never',
    'exercise_freq': 'Moderate',
    'family_history': 0,
    'diabetes': 0,
    'heart_disease': 0
}

result = run_assessment(my_patient)

🏥 HEALTH RISK ASSESSMENT

[1/4] 🔍 Validating input...
✅ Input valid

[2/4] 🤖 Running prediction...
✅ Risk: Low

[3/4] 🔎 Finding similar patients...
   • P0362: Low risk (distance: 0.02)
   • P0412: High risk (distance: 0.02)
   • P0336: Low risk (distance: 0.03)

[4/4] 📝 Generating explanation...


┌──────────────────────────────────────────────────────────────┐
│              HEALTH RISK ASSESSMENT                          │
├──────────────────────────────────────────────────────────────┤
│  Risk Level: Low        │ Score:  50.0%              │
│  ⚠️ Low confidence (50.0%)                                    │
├──────────────────────────────────────────────────────────────┤
│  Top Contributing Factors:                                   │
│    • cholesterol: 0.169                                     │
│    • bmi: 0.156                                             │
│    • systolic_bp: 0.150                                     │
│    • glucose: 0.123                                       